In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / "src"))

from utils import load_dataset, ensure_sorted, train_test_split_time_series
from feature_engineering import add_lag_features, add_rolling_features, add_time_features
from evaluation import evaluate_forecast, print_evaluation
from models.arima_model import run_arima
from models.sarima_model import run_sarima
from models.sarimax_model import run_sarimax


In [ ]:
# Load merged dataset
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
MERGED_PATH = PROCESSED_DIR / "demand_temperature_half_hourly.csv"

df = load_dataset(MERGED_PATH)
df = ensure_sorted(df)

# Feature engineering
df_fe = df.copy()

df_fe = add_lag_features(df_fe, column='demand', lags=[1, 48, 96])
df_fe = add_rolling_features(df_fe, column='demand', windows=[48, 96, 336])
df_fe = add_time_features(df_fe)

# Drop rows created by lag/rolling
df_fe = df_fe.dropna().reset_index(drop=True)




In [ ]:
# Train/test split
train, test = train_test_split_time_series(df_fe, test_size=0.1)

# Prepare features and target
features = [c for c in df_fe.columns if c not in ['timestamp', 'demand']]
X_train, y_train = train[features], train['demand']
X_test, y_test = test[features], test['demand']

train.head(), test.head()

3 — Naive baseline

In [ ]:
y_pred_naive = test['demand'].shift(1).fillna(method='bfill')
metrics_naive = evaluate_forecast(y_test, y_pred_naive)
print("Naive baseline:")
print_evaluation(metrics_naive)


4 — ARIMA

In [ ]:
train_series = train['demand']

arima_model = run_arima(train_series, order=(5,1,0))
arima_forecast = arima_model.forecast(steps=len(test))

metrics_arima = evaluate_forecast(y_test, arima_forecast)
print("ARIMA:")
print_evaluation(metrics_arima)


5 — SARIMA

In [ ]:
sarima_model = run_sarima(train_series, order=(2,1,2), seasonal_order=(1,1,1,48))
sarima_forecast = sarima_model.forecast(steps=len(test))

metrics_sarima = evaluate_forecast(y_test, sarima_forecast)
print("SARIMA:")
print_evaluation(metrics_sarima)


6 — SARIMAX (with temperature)

In [ ]:
train_y = train['demand']
train_exog = train[['tmean']]
test_exog = test[['tmean']]

sarimax_model = run_sarimax(train_y, train_exog,
                            order=(2,1,2), seasonal_order=(1,1,1,48))
sarimax_forecast = sarimax_model.forecast(steps=len(test), exog=test_exog)

metrics_sarimax = evaluate_forecast(y_test, sarimax_forecast)
print("SARIMAX (with temperature):")
print_evaluation(metrics_sarimax)


7 — Comparison table

In [ ]:
results = pd.DataFrame([
    {'Model': 'Naive', **metrics_naive},
    {'Model': 'ARIMA', **metrics_arima},
    {'Model': 'SARIMA', **metrics_sarima},
    {'Model': 'SARIMAX', **metrics_sarimax},
])

results


8 — Plot best model vs actual

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(test['timestamp'], y_test, label='Actual')

plt.plot(test['timestamp'], sarimax_forecast, label='SARIMAX', alpha=0.8)

plt.legend()
plt.title("Best Model vs Actual")
plt.tight_layout()
plt.show()
